In [1]:
from flwr_datasets.partitioner import IidPartitioner
from flwr_datasets import FederatedDataset


partitioner = IidPartitioner(num_partitions=10)
FDS = FederatedDataset(
    dataset="vicgalle/alpaca-gpt4",
    partitioners={"train": partitioner},
)
client_trainset = FDS.load_partition(1, "train")
client_trainset = client_trainset.rename_column("output", "response")
client_trainset

ModuleNotFoundError: No module named 'flwr_datasets'

In [3]:
type(client_trainset)

datasets.arrow_dataset.Dataset

In [20]:
import datasets
from datasets import load_dataset, DatasetDict
import pandas as pd
from functools import partial
from sklearn.model_selection import train_test_split


def get_dataset(dataset_name, local_data_dir=None):

    if dataset_name in ["gsm8k"]:
        dataset_name = local_data_dir + dataset_name if local_data_dir is not None else dataset_name
        dataset = load_dataset(dataset_name, name="main")
    elif dataset_name in ["lighteval/MATH"]:
        dataset_name = local_data_dir + dataset_name if local_data_dir is not None else dataset_name
        dataset = load_dataset(dataset_name, name="all")
    else:
        dataset_name = local_data_dir + dataset_name if local_data_dir is not None else dataset_name
        dataset = load_dataset(dataset_name)

    return dataset

# Function to split a dataset dictionary into two 50/50 parts
def split_dataset_50_50(dataset_dict):
    split_datasets = {
        'ds_1': None,
        'ds_2': None
    }
    for split in ['train', 'valid', 'test']:
        if split in dataset_dict:
            dataset_split_1, dataset_split_2 = train_test_split(
                dataset_dict[split], test_size=0.5, shuffle=True, seed=42
            )
            print(f">> ===== After split, Dataset1 {split} has {len(dataset_split_1)} examples. =====")
            print(f">> ===== After split, Dataset2 {split} has {len(dataset_split_2)} examples. =====")
            split_datasets['ds_1'][split] = dataset_split_1
            split_datasets['ds_2'][split] = dataset_split_2
    return DatasetDict(split_datasets['ds_1']), DatasetDict(split_datasets['ds_2'])


def process_sft_dataset(dataset_name, dataset, dataset_sample):
    if dataset_name in ["lucasmccabe-lmi/CodeAlpaca-20k", "yahma/alpaca-cleaned", "FinGPT/fingpt-sentiment-train"]:
        dataset = dataset.map(alpaca_format, remove_columns=['input', 'output'], desc=f"Preprocessing {dataset_name} for unified format.")
    elif dataset_name in ["WizardLM/WizardLM_evol_instruct_70k"]:
        dataset = dataset.rename_column("output", "response")
    elif dataset_name in ["tatsu-lab/alpaca", "vicgalle/alpaca-gpt4", "gbharti/finance-alpaca"]:
        dataset = dataset.map(alpaca_format, remove_columns=['input', 'output', 'text'], desc=f"Preprocessing {dataset_name} for unified format.")
    elif dataset_name in ["TIGER-Lab/MathInstruct"]:
        df = pd.DataFrame(dataset)
        df = df.drop_duplicates(subset=['instruction'])
        dataset = datasets.Dataset.from_pandas(df)
        dataset = dataset.rename_column("output", "response")
        dataset = dataset.remove_columns(['source'])
    elif dataset_name in ["lighteval/MATH"]:
        dataset = dataset.rename_column("solution", "response")
        dataset = dataset.rename_column("problem", "instruction")
        dataset = dataset.remove_columns(['level', 'type'])
    elif dataset_name in ['gsm8k']:
        dataset = dataset.rename_column("question", "instruction")
        dataset = dataset.rename_column("answer", "response")
    elif dataset_name in ['medalpaca/medical_meadow_medical_flashcards']:       # TODO: 'lavita/ChatDoctor-HealthCareMagic-100k'. not sure whether to discard the instruction.
        dataset = dataset.remove_columns(['instruction'])
        dataset = dataset.rename_column("input", "instruction")
        dataset = dataset.rename_column("output", "response")
    else:
        raise NotImplementedError(f"Dataset {dataset_name} is not supported.")
    dataset = dataset.shuffle(seed=2023)
    if dataset_sample:
        num_sample = min(len(dataset), dataset_sample)
        dataset = dataset.select(range(num_sample))
    print(f">> ===== After processing, Dataset {dataset_name} has {len(dataset)} examples. =====")
    print(f">> ===== Spliting two parts datasets =====")
    
    if len(dataset['train']) > 10000 and len(dataset['test']) >= 2000:
        dataset = split_dataset_50_50(dataset)
    return dataset

def alpaca_format(example):
    if example['input'] == "":
        example["instruction"] = example["instruction"]
    else:
        example["instruction"] = example["instruction"] + " " + example['input']
    example["response"] = example['output']
    return example

    
def find_common_prefix(str1, str2):
    prefix = ""
    for i in range(min(len(str1), len(str2))):
        if str1[i] == str2[i]:
            prefix += str1[i]
        else:
            break
    return prefix




In [21]:
dataset = get_dataset(dataset_name="gsm8k", local_data_dir=None)
datasets = process_sft_dataset(dataset_name="gsm8k", dataset=dataset, dataset_sample=None)
datasets

>> ===== After processing, Dataset gsm8k has 2 examples. =====
>> ===== Spliting two parts datasets =====


DatasetDict({
    train: Dataset({
        features: ['instruction', 'response'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['instruction', 'response'],
        num_rows: 1319
    })
})

In [2]:
import torch
torch.cuda.is_available()

True

In [4]:
from flwr.common import Context, Message, MessageType, ConfigsRecord
from flwr.client.typing import ClientAppCallable
from typing import Callable
import wandb
import time

# Define type alias for Mod
Mod = Callable[[Message, Context, ClientAppCallable], Message]

SyntaxError: invalid syntax (99868249.py, line 8)

In [18]:
def clean_llm_text(text):
    """
    Clean and normalize text from LLM outputs by removing noise and repetitions.
    
    Args:
        text (str): Raw text from LLM prediction
        
    Returns:
        str: Cleaned and normalized text
    """
    import re
    
    # Remove repetitive patterns (like 'cor cor cor' or 'asesases')
    def remove_repetitions(text):
        # Split into words
        words = text.split()
        cleaned_words = []
        prev_word = None
        repetition_count = 0
        
        for word in words:
            if word == prev_word:
                repetition_count += 1
                if repetition_count < 2:  # Allow up to 2 repetitions for legitimate cases
                    cleaned_words.append(word)
            else:
                repetition_count = 0
                cleaned_words.append(word)
            prev_word = word
            
        return ' '.join(cleaned_words)
    
    def remove_repeats(text):
        # Remove repeated words
        pattern_words = r'\b(\w+)(?:\s+\1\b)+'
        text = re.sub(pattern_words, r'\1', text)

        # Remove repeated character patterns (like 'asasas')
        pattern_chars = r'(\w+?)\1+'
        text = re.sub(pattern_chars, r'\1', text)

        return text
    
    # Remove excessive punctuation
    def normalize_punctuation(text):
        # Replace multiple exclamation/question marks with single ones
        text = re.sub(r'!+', '!', text)
        text = re.sub(r'\?+', '?', text)
        # Remove multiple periods (except for ellipsis)
        text = re.sub(r'\.{4,}', '...', text)
        text = text.replace('cor', '').replace('asesa', '')
        return text
    
    # Main cleaning pipeline
    cleaned_text = text.strip()
    
    # Remove common noise patterns
    noise_patterns = [
        r'\n+',              # Multiple newlines
        r'\s+',              # Multiple spaces
        r'\\n',              # Literal \n
        r'\\t',              # Literal \t
    ]
    
    for pattern in noise_patterns:
        cleaned_text = re.sub(pattern, ' ', cleaned_text)
    
    # Apply cleaning functions
    # cleaned_text = remove_repetitions(cleaned_text)
    cleaned_text = remove_repeats(cleaned_text)
    cleaned_text = normalize_punctuation(cleaned_text)
    cleaned_text = ' '.join(cleaned_text.split())  # Normalize spacing
    
    return cleaned_text

In [20]:
text_str = f"""at\n\nSea Animals: Whale, Fish cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor', " cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor One example of a technology that uses artificial intelligence is a virtual personal assistant such as Amazon's Alexa, Apple's Siri, or Google Assistant. These devices use natural language processing and machine learning to understand and respond to user's voice commands, providing assistance in tasks such as setting reminders, playing music, or answering questions. cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor cor"""
text_str1 = f"""following into their based mammals animals, water animals.\n\n1 Response:\n \n animals: \n, Elephant,Sea Animals: Dolphin, Dolphin\n\nasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesa"""
clean_llm_text(text_str1)

'folowing into their based mamals animals, water animals. 1 Response: animals: , Elephant,Sea Animals: Dolphin, Dolphin'

In [23]:
print(text_str1)

following into their based mammals animals, water animals.

1 Response:
 
 animals: 
, Elephant,Sea Animals: Dolphin, Dolphin

asesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesasesa